In [1]:
import os
import json
from pathlib import Path

import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

In [2]:
MONGO_URI = "mongodb://localhost:27017"

try:
    mongo_client = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=5000
    )

    mongo_client.admin.command("ping")

    db = mongo_client["kohler_ai_bathroom"]
    products_collection = db["products"]

    print("✓ MongoDB connected successfully.")
    print("Database:", db.name)
    print("Collection:", products_collection.name)

except Exception as e:
    raise ConnectionError(
        "Could not connect to MongoDB. "
        "Make sure MongoDB is running."
    ) from e

✓ MongoDB connected successfully.
Database: kohler_ai_bathroom
Collection: products


In [3]:
total_products = products_collection.count_documents({})

print("Total products in MongoDB:", total_products)

Total products in MongoDB: 36


In [4]:
products = list(
    products_collection.find(
        {},
        {"_id": 0}
    )
)

print(f"Loaded {len(products)} products.")

Loaded 36 products.


In [5]:
df = pd.DataFrame(products)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (36, 23)


,product_id,category,collection,color,dimensions,electrical,extraction_method,extraction_status,features,installation,...,source_type,subcategory,validation_problems,category_original,subcategory_original,installation_type,waste_outlet,voltage,spatial_validation_possible,data_quality
0,1408991-IN4-A,Other,Not specified,[White],"{'width_mm': 344.0, 'width_mm_available': True...","{'required': False, 'required_source': 'inferr...",Groq text extraction,success,[Ergonomic and Straight line design suitable f...,"{'type': 'Above Counter', 'waste_outlet': 'Not...",...,KOHLER specification PDF,Not specified,[Missing category],NaN,NaN,Not specified,Not specified,Not specified,True,complete_for_spatial_use
1,29024IN-1,Vessel,Not specified,[White],"{'width_mm': 470.0, 'width_mm_available': True...","{'required': False, 'required_source': 'inferr...",Groq text extraction,success,"[Vitreous China, Circular Design, Without Over...","{'type': 'Above counter', 'waste_outlet': 'Not...",...,KOHLER specification PDF,Vessel,[],Vessel,Vessel,Not specified,Not specified,Not specified,True,complete_for_spatial_use
2,1311520-A04-D,BASINS / COUNTERTOP,Not specified,"[white, honed black]","{'width_mm': 393.0, 'width_mm_available': True...","{'required': False, 'required_source': 'inferr...",Groq text extraction,success,"[Bench top installation, Slim vessel, CleanCoa...","{'type': 'Bench top installation', 'waste_outl...",...,KOHLER specification PDF,Countertop Basin,[Missing height],BASINS / COUNTERTOP,Countertop Basin,Not specified,Not specified,Not specified,False,complete_with_unspecified_source_fields
3,K-12927IN,Other,Not specified,"[Polished Chrome, Vibrant® French Gold, Vibran...","{'width_mm': -1.0, 'width_mm_available': False...","{'required': False, 'required_source': 'inferr...",Groq text extraction,success,[Designed to coordinate with a range of KOHLER...,"{'type': 'Wall-mount', 'waste_outlet': 'Not sp...",...,KOHLER specification PDF,Not specified,"[Missing category, Missing width, Missing dept...",NaN,NaN,Not specified,Not specified,Not specified,False,complete_with_unspecified_source_fields
4,K-1381T-S,Toilet,Veil,[White],"{'width_mm': -1.0, 'width_mm_available': False...","{'required': False, 'required_source': 'inferr...",Groq text extraction,success,[One-piece toilets integrate the tank and bowl...,"{'type': 'Floor-mount', 'waste_outlet': 'Floor...",...,KOHLER specification PDF,One-piece elongated toilet,"[Missing width, Missing depth]",Toilet,One-piece elongated toilet,Not specified,Not specified,Not specified,False,complete_with_unspecified_source_fields


In [6]:
all_fields = set()

for product in products:
    all_fields.update(product.keys())

print("Fields found in the dataset:\n")

for field in sorted(all_fields):
    print("-", field)

Fields found in the dataset:

- category
- category_original
- collection
- color
- data_quality
- dimensions
- electrical
- extraction_method
- extraction_status
- features
- installation
- installation_type
- material
- product_id
- product_name
- source_pdf
- source_type
- spatial_validation_possible
- subcategory
- subcategory_original
- validation_problems
- voltage
- waste_outlet


In [7]:
def get_nested_value(product, parent, child):
    value = product.get(parent)

    if isinstance(value, dict):
        return value.get(child)

    return None

In [8]:
inspection_data = []

for product in products:

    inspection_data.append({
        "product_id": product.get("product_id"),
        "product_name": product.get("product_name"),
        "category": product.get("category"),
        "subcategory": product.get("subcategory"),
        "collection": product.get("collection"),

        "width_mm": get_nested_value(
            product, "dimensions", "width_mm"
        ),

        "depth_mm": get_nested_value(
            product, "dimensions", "depth_mm"
        ),

        "height_mm": get_nested_value(
            product, "dimensions", "height_mm"
        ),

        "rough_in_mm": get_nested_value(
            product, "installation", "rough_in_mm"
        ),

        "installation_type": get_nested_value(
            product, "installation", "type"
        ),

        "waste_outlet": get_nested_value(
            product, "installation", "waste_outlet"
        ),

        "electrical_required": get_nested_value(
            product, "electrical", "required"
        ),

        "voltage": get_nested_value(
            product, "electrical", "voltage"
        ),

        "power_w": get_nested_value(
            product, "electrical", "power_w"
        ),

        "material": product.get("material"),
        "color": product.get("color"),
        "features": product.get("features")
    })

inspection_df = pd.DataFrame(inspection_data)

display(inspection_df)

,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,rough_in_mm,installation_type,waste_outlet,electrical_required,voltage,power_w,material,color,features
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),Other,Not specified,Not specified,344.0,483.0,141.0,-1.0,Above Counter,Not specified,False,Not specified,-1.0,Vitreous China,[White],[Ergonomic and Straight line design suitable f...
1,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Vessel,Vessel,Not specified,470.0,116.0,139.0,-1.0,Above counter,Not specified,False,Not specified,-1.0,Vitreous China,[White],"[Vitreous China, Circular Design, Without Over..."
2,1311520-A04-D,Mica® Square 393mm Basin,BASINS / COUNTERTOP,Countertop Basin,Not specified,393.0,393.0,-1.0,50.0,Bench top installation,DN32 waste without overflow,False,Not specified,-1.0,Ceramic,"[white, honed black]","[Bench top installation, Slim vessel, CleanCoa..."
3,K-12927IN,Complimentary™ Hygiene Spray,Other,Not specified,Not specified,-1.0,-1.0,-1.0,-1.0,Wall-mount,Not specified,False,Not specified,-1.0,Not specified,"[Polished Chrome, Vibrant® French Gold, Vibran...",[Designed to coordinate with a range of KOHLER...
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,One-piece elongated toilet,Veil,-1.0,-1.0,390.0,305.0,Floor-mount,Floor,False,Not specified,-1.0,Plastic seat,[White],[One-piece toilets integrate the tank and bowl...
5,K-17629T-NS,Ove™ One-piece round-front toilet with skirted...,Toilet,One-piece toilet,Ove™,-1.0,-1.0,390.0,185.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],[One-piece toilets integrate the tank and bowl...
6,K-17660T-M,Ove™ Quiet-Close™ elongated toilet seat,Toilet Seat,Not specified,Not specified,-1.0,-1.0,-1.0,-1.0,Not specified,Not specified,False,Not specified,-1.0,Plastic construction,[White],"[Elongated closed-front seat with lid, Quiet-C..."
7,K-1851IN,Brive Plus,Other,Not specified,Not specified,-1.0,-1.0,395.0,180.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],[Round-front bowl offers an ideal solution for...
8,K-1853IN,Brive Plus,Other,Not specified,Not specified,-1.0,-1.0,395.0,220.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],"[Elongated bowl offers added room and comfort,..."
9,K-21226IN,ModernLife Edge 600 mm rectangular vessel bath...,Sink,Vessel sink,ModernLife Edge,600.0,113.0,-1.0,-1.0,Vessel,Not specified,False,Not specified,-1.0,Vitreous china,"[Honed Black, Thunder grey, Honed Peacock, Hon...","[Contemporary design makes a modern statement,..."


In [9]:
missing_counts = inspection_df.isna().sum()

missing_percentage = (
    inspection_df.isna().mean() * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentage
})

display(
    missing_report.sort_values(
        "missing_count",
        ascending=False
    )
)

,missing_count,missing_percentage
product_id,0,0.0
product_name,0,0.0
category,0,0.0
subcategory,0,0.0
collection,0,0.0
width_mm,0,0.0
depth_mm,0,0.0
height_mm,0,0.0
rough_in_mm,0,0.0
installation_type,0,0.0


In [10]:
missing_product_ids = inspection_df[
    inspection_df["product_id"].isna()
    | (inspection_df["product_id"].astype(str).str.strip() == "")
]

print(
    "Products with missing product_id:",
    len(missing_product_ids)
)

display(missing_product_ids)

Products with missing product_id: 0


,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,rough_in_mm,installation_type,waste_outlet,electrical_required,voltage,power_w,material,color,features


In [11]:
duplicate_ids = inspection_df[
    inspection_df["product_id"].duplicated(
        keep=False
    )
]

print(
    "Products involved in duplicate product IDs:",
    len(duplicate_ids)
)

display(
    duplicate_ids.sort_values("product_id")
)

Products involved in duplicate product IDs: 0


,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,rough_in_mm,installation_type,waste_outlet,electrical_required,voltage,power_w,material,color,features


In [12]:
missing_names = inspection_df[
    inspection_df["product_name"].isna()
    | (inspection_df["product_name"].astype(str).str.strip() == "")
]

print(
    "Products with missing product_name:",
    len(missing_names)
)

display(missing_names)

Products with missing product_name: 0


,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,rough_in_mm,installation_type,waste_outlet,electrical_required,voltage,power_w,material,color,features


In [13]:
print("Categories found:\n")

print(
    inspection_df["category"]
    .value_counts(dropna=False)
)

Categories found:

category
Faucet                  14
Toilet                   5
Bathroom sink            5
Other                    4
Sink                     2
Vessel                   1
BASINS / COUNTERTOP      1
Toilet Seat              1
Bathroom Sink            1
Faucet Trim              1
Bathroom sink faucet     1
Name: count, dtype: int64


In [14]:
print("Subcategories found:\n")

print(
    inspection_df["subcategory"]
    .value_counts(dropna=False)
)

Subcategories found:

subcategory
Not specified                              10
Vessel sink                                 5
Vessel                                      4
Bathroom sink faucet                        3
Health faucet                               3
Single-handle bathroom sink faucet          2
Countertop Basin                            1
One-piece elongated toilet                  1
One-piece toilet                            1
Wall-mount bathroom sink faucet             1
Tall single-handle bathroom sink faucet     1
One-piece round-front smart toilet          1
One-piece round-front toilet                1
Wall-mount bathroom sink faucet trim        1
Single-handle                               1
Name: count, dtype: int64


In [15]:
dimension_columns = [
    "width_mm",
    "depth_mm",
    "height_mm"
]

dimension_missing = inspection_df[
    inspection_df[dimension_columns]
    .isna()
    .any(axis=1)
]

print(
    "Products with at least one missing dimension:",
    len(dimension_missing)
)

display(
    dimension_missing[
        [
            "product_id",
            "product_name",
            "category",
            "width_mm",
            "depth_mm",
            "height_mm"
        ]
    ]
)

Products with at least one missing dimension: 0


,product_id,product_name,category,width_mm,depth_mm,height_mm


In [16]:
invalid_dimensions = inspection_df[
    (
        inspection_df["width_mm"].notna()
        & (inspection_df["width_mm"] <= 0)
    )
    |
    (
        inspection_df["depth_mm"].notna()
        & (inspection_df["depth_mm"] <= 0)
    )
    |
    (
        inspection_df["height_mm"].notna()
        & (inspection_df["height_mm"] <= 0)
    )
]

print(
    "Products with invalid/non-positive dimensions:",
    len(invalid_dimensions)
)

display(invalid_dimensions)

Products with invalid/non-positive dimensions: 34


,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,rough_in_mm,installation_type,waste_outlet,electrical_required,voltage,power_w,material,color,features
2,1311520-A04-D,Mica® Square 393mm Basin,BASINS / COUNTERTOP,Countertop Basin,Not specified,393.0,393.0,-1.0,50.0,Bench top installation,DN32 waste without overflow,False,Not specified,-1.0,Ceramic,"[white, honed black]","[Bench top installation, Slim vessel, CleanCoa..."
3,K-12927IN,Complimentary™ Hygiene Spray,Other,Not specified,Not specified,-1.0,-1.0,-1.0,-1.0,Wall-mount,Not specified,False,Not specified,-1.0,Not specified,"[Polished Chrome, Vibrant® French Gold, Vibran...",[Designed to coordinate with a range of KOHLER...
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,One-piece elongated toilet,Veil,-1.0,-1.0,390.0,305.0,Floor-mount,Floor,False,Not specified,-1.0,Plastic seat,[White],[One-piece toilets integrate the tank and bowl...
5,K-17629T-NS,Ove™ One-piece round-front toilet with skirted...,Toilet,One-piece toilet,Ove™,-1.0,-1.0,390.0,185.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],[One-piece toilets integrate the tank and bowl...
6,K-17660T-M,Ove™ Quiet-Close™ elongated toilet seat,Toilet Seat,Not specified,Not specified,-1.0,-1.0,-1.0,-1.0,Not specified,Not specified,False,Not specified,-1.0,Plastic construction,[White],"[Elongated closed-front seat with lid, Quiet-C..."
7,K-1851IN,Brive Plus,Other,Not specified,Not specified,-1.0,-1.0,395.0,180.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],[Round-front bowl offers an ideal solution for...
8,K-1853IN,Brive Plus,Other,Not specified,Not specified,-1.0,-1.0,395.0,220.0,Floor-mount,Floor,False,Not specified,-1.0,Not specified,[White],"[Elongated bowl offers added room and comfort,..."
9,K-21226IN,ModernLife Edge 600 mm rectangular vessel bath...,Sink,Vessel sink,ModernLife Edge,600.0,113.0,-1.0,-1.0,Vessel,Not specified,False,Not specified,-1.0,Vitreous china,"[Honed Black, Thunder grey, Honed Peacock, Hon...","[Contemporary design makes a modern statement,..."
10,K-2200IN,Conical Bell™,Sink,Vessel sink,Not specified,413.0,-1.0,-1.0,464.0,Semi-recessed vessel,"Center drain, no overflow",False,Not specified,-1.0,Vitreous china,"[White, Cashmere, Black]","[Traditional design creates a classic look, Ro..."
11,K-23486IN-4ND,"Parallel™ Wall-mount bathroom sink faucet, 9.0...",Faucet,Wall-mount bathroom sink faucet,Parallel,-1.0,-1.0,-1.0,-1.0,Wall-mount,Not specified,False,Not specified,-1.0,Premium metal,"[Polished Chrome, Vibrant French Gold, Vibrant...",[Single handle controls both on/off activation...


In [17]:
installation_columns = [
    "rough_in_mm",
    "installation_type",
    "waste_outlet"
]

display(
    inspection_df[
        [
            "product_id",
            "product_name",
            "category"
        ] + installation_columns
    ]
)

,product_id,product_name,category,rough_in_mm,installation_type,waste_outlet
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),Other,-1.0,Above Counter,Not specified
1,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Vessel,-1.0,Above counter,Not specified
2,1311520-A04-D,Mica® Square 393mm Basin,BASINS / COUNTERTOP,50.0,Bench top installation,DN32 waste without overflow
3,K-12927IN,Complimentary™ Hygiene Spray,Other,-1.0,Wall-mount,Not specified
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,305.0,Floor-mount,Floor
5,K-17629T-NS,Ove™ One-piece round-front toilet with skirted...,Toilet,185.0,Floor-mount,Floor
6,K-17660T-M,Ove™ Quiet-Close™ elongated toilet seat,Toilet Seat,-1.0,Not specified,Not specified
7,K-1851IN,Brive Plus,Other,180.0,Floor-mount,Floor
8,K-1853IN,Brive Plus,Other,220.0,Floor-mount,Floor
9,K-21226IN,ModernLife Edge 600 mm rectangular vessel bath...,Sink,-1.0,Vessel,Not specified


In [18]:
for product in products:

    print("=" * 80)

    print("Product ID:", product.get("product_id"))
    print("Product:", product.get("product_name"))
    print("Category:", product.get("category"))

    print("\nFeatures:")

    features = product.get("features", [])

    if features:
        for feature in features:
            print(" -", feature)
    else:
        print(" - No features extracted")

Product ID: 1408991-IN4-A
Product: SPAN® Square Vessel Without Deck ( Small)
Category: Other

Features:
 - Ergonomic and Straight line design suitable for compact rooms with deck space.
 - Easy installation as it requires counter cutting to accommodate drain only.
 - Optimum Depth to contain Splashes.
 - Above counter without faucet deck.
 - Only drain cutting (no profile cutting required).
 - Without Overflow Hole
Product ID: 29024IN-1
Product: CHALICE ROUND VESSEL 1TAP HOLE
Category: Vessel

Features:
 - Vitreous China
 - Circular Design
 - Without Overflow
 - Above counter
 - Only Drain cutting (no profile cutting required)
Product ID: 1311520-A04-D
Product: Mica® Square 393mm Basin
Category: BASINS / COUNTERTOP

Features:
 - Bench top installation
 - Slim vessel
 - CleanCoatTM - stain resistant & easy to clean
Product ID: K-12927IN
Product: Complimentary™ Hygiene Spray
Category: Other

Features:
 - Designed to coordinate with a range of KOHLER faucets
 - Thread = G 1/2
 - Includes 

In [19]:
for product in products:

    color = product.get("color")

    print(
        product.get("product_id"),
        "→",
        color
    )

1408991-IN4-A → ['White']
29024IN-1 → ['White']
1311520-A04-D → ['white', 'honed black']
K-12927IN → ['Polished Chrome', 'Vibrant® French Gold', 'Vibrant® Brushed Nickel']
K-1381T-S → ['White']
K-17629T-NS → ['White']
K-17660T-M → ['White']
K-1851IN → ['White']
K-1853IN → ['White']
K-21226IN → ['Honed Black', 'Thunder grey', 'Honed Peacock', 'Honed Lush']
K-2200IN → ['White', 'Cashmere', 'Black']
K-23486IN-4ND → ['Polished Chrome', 'Vibrant French Gold', 'Vibrant Brushed Bronze', 'Matte Black', 'Vibrant Rose Gold', 'Vibrant Brushed Rose Gold']
K-23966IN-4ND → ['Polished Chrome', 'Vibrant® French Gold', 'Matte Black', 'Vibrant Rose Gold']
K-23967IN-4ND → ['Polished Chrome', 'Vibrant® French Gold', 'Matte Black', 'Vibrant Rose Gold']
K-25316IN → ['White']
K-2661IN → ['White', 'Black']
K-27477IN-4ND → ['Polished Chrome']
K-27480IN-4ND → ['Polished Chrome', 'Vibrant® French Gold']
K-27485IN-4 → ['Polished Chrome', 'Vibrant® French Gold']
K-28529IN → ['White']
K-28784IN → ['White', 'Cashmer

In [20]:
validation_issues = []

for product in products:

    problems = product.get(
        "validation_problems",
        []
    )

    if problems:

        validation_issues.append({
            "product_id": product.get("product_id"),
            "product_name": product.get("product_name"),
            "problems": problems
        })

print(
    "Products with validation warnings:",
    len(validation_issues)
)

display(
    pd.DataFrame(validation_issues)
)

Products with validation warnings: 35


,product_id,product_name,problems
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),[Missing category]
1,1311520-A04-D,Mica® Square 393mm Basin,[Missing height]
2,K-12927IN,Complimentary™ Hygiene Spray,"[Missing category, Missing width, Missing dept..."
3,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,"[Missing width, Missing depth]"
4,K-17629T-NS,Ove™ One-piece round-front toilet with skirted...,"[Missing width, Missing depth]"
5,K-17660T-M,Ove™ Quiet-Close™ elongated toilet seat,"[Missing width, Missing depth, Missing height]"
6,K-1851IN,Brive Plus,"[Missing category, Missing width, Missing depth]"
7,K-1853IN,Brive Plus,"[Missing category, Missing width, Missing depth]"
8,K-21226IN,ModernLife Edge 600 mm rectangular vessel bath...,[Missing height]
9,K-2200IN,Conical Bell™,"[Missing depth, Missing height]"


In [21]:
print(
    inspection_df.shape
)

print("\nExtraction status:")

status_counts = pd.DataFrame(products)[
    "extraction_status"
].value_counts(dropna=False)

display(status_counts)

(36, 17)

Extraction status:


extraction_status
success    36
Name: count, dtype: int64

In [22]:
important_fields = [
    "product_id",
    "product_name",
    "category"
]

suspicious_products = []

for product in products:

    missing_important = []

    for field in important_fields:

        value = product.get(field)

        if value is None or str(value).strip() == "":
            missing_important.append(field)

    if missing_important:

        suspicious_products.append({
            "product_id": product.get("product_id"),
            "product_name": product.get("product_name"),
            "missing_important_fields": missing_important,
            "source_pdf": product.get("source_pdf")
        })

print(
    "Suspicious products:",
    len(suspicious_products)
)

display(
    pd.DataFrame(suspicious_products)
)

Suspicious products: 0


""


In [23]:
inspection_path = Path(
    "kohler_data_cleaning_inspection.csv"
)

inspection_df.to_csv(
    inspection_path,
    index=False
)

print(
    "Inspection report saved to:"
)

print(
    inspection_path.resolve()
)

Inspection report saved to:
C:\Users\Abhist\Desktop\KOHLER\scripts\kohler_data_cleaning_inspection.csv


In [24]:
# Backup the current extracted products

backup_collection = db["products_backup"]

# Remove an old backup if you previously created one
backup_collection.delete_many({})

# Copy all current products
current_products = list(
    products_collection.find({})
)

if current_products:
    backup_collection.insert_many(current_products)

print(
    f"Backup created successfully: {len(current_products)} products"
)

Backup created successfully: 36 products


# Final Missing-Value Cleaning

This section converts missing values into explicit, safe placeholders rather than inventing product specifications.

**Important:** `-1` means the numeric specification was not available in the source data. It is NOT a real product dimension. The corresponding `*_available` flag tells the recommendation/layout engine whether that value may be used for calculations.

Text fields use `Not specified`, list fields use `['Not specified']`, and missing categories use `Other`. Original values are preserved in `category_original` and `subcategory_original` where useful.

In [25]:
# ============================================
# 1. RELOAD PRODUCTS BEFORE CLEANING
# ============================================

products = list(products_collection.find({}, {"_id": 0}))

print(f"Products loaded for cleaning: {len(products)}")
assert len(products) > 0, "No products found in MongoDB."


Products loaded for cleaning: 36


In [26]:
# ============================================
# 2. CLEANING RULES
# ============================================

# Numeric fields use -1 when the source does not provide a value.
# NEVER interpret -1 as a real dimension/specification.
NUMERIC_FIELDS = [
    "width_mm",
    "depth_mm",
    "height_mm",
    "rough_in_mm",
    "power_w"
]

TEXT_FIELDS = [
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "collection",
    "installation_type",
    "waste_outlet",
    "voltage",
    "material"
]

LIST_FIELDS = [
    "color",
    "features"
]

print("Cleaning rules loaded.")


Cleaning rules loaded.


In [27]:
# ============================================
# 3. HELPER FUNCTIONS
# ============================================

def is_blank(value):
    """Return True for None, NaN, empty strings, or empty containers."""
    if value is None:
        return True

    if isinstance(value, float) and pd.isna(value):
        return True

    if isinstance(value, str) and value.strip() == "":
        return True

    if isinstance(value, (list, dict)) and len(value) == 0:
        return True

    return False


def clean_text(value):
    """Clean text while preserving the fact that the source did not specify it."""
    if is_blank(value):
        return "Not specified"
    return str(value).strip()


def clean_numeric(value):
    """Convert numeric values to float; use -1 for unavailable source values."""
    if is_blank(value):
        return -1.0

    try:
        number = float(value)
        return number if number > 0 else -1.0
    except (TypeError, ValueError):
        return -1.0


def clean_list(value):
    """Ensure list fields never contain null/empty values."""
    if is_blank(value):
        return ["Not specified"]

    if not isinstance(value, list):
        value = [value]

    cleaned = []
    for item in value:
        if not is_blank(item):
            cleaned.append(str(item).strip())

    return cleaned if cleaned else ["Not specified"]


In [28]:
# ============================================
# 4. CLEAN EACH PRODUCT
# ============================================

cleaned_products = []

for product in products:
    p = dict(product)

    # Preserve the original category/subcategory before cleaning.
    p["category_original"] = product.get("category")
    p["subcategory_original"] = product.get("subcategory")

    # ----------------------------------------
    # Top-level text fields
    # ----------------------------------------
    for field in TEXT_FIELDS:
        p[field] = clean_text(product.get(field))

    # Category fallback: explicit unknown category.
    if p["category"] == "Not specified":
        p["category"] = "Other"

    # ----------------------------------------
    # List fields
    # ----------------------------------------
    for field in LIST_FIELDS:
        p[field] = clean_list(product.get(field))

    # ----------------------------------------
    # Dimensions
    # ----------------------------------------
    dimensions = product.get("dimensions")
    if not isinstance(dimensions, dict):
        dimensions = {}

    cleaned_dimensions = {}
    for field in ["width_mm", "depth_mm", "height_mm"]:
        cleaned_dimensions[field] = clean_numeric(dimensions.get(field))
        cleaned_dimensions[field + "_available"] = (
            cleaned_dimensions[field] > 0
        )

    p["dimensions"] = cleaned_dimensions

    # ----------------------------------------
    # Installation
    # ----------------------------------------
    installation = product.get("installation")
    if not isinstance(installation, dict):
        installation = {}

    p["installation"] = {
        "type": clean_text(installation.get("type")),
        "waste_outlet": clean_text(installation.get("waste_outlet")),
        "rough_in_mm": clean_numeric(installation.get("rough_in_mm")),
        "rough_in_available": clean_numeric(installation.get("rough_in_mm")) > 0
    }

    # ----------------------------------------
    # Electrical
    # ----------------------------------------
    electrical = product.get("electrical")
    if not isinstance(electrical, dict):
        electrical = {}

    voltage = electrical.get("voltage")
    power = clean_numeric(electrical.get("power_w"))

    # If the source explicitly says electrical_required, preserve it.
    # Otherwise infer True only when voltage or power is explicitly present.
    required = electrical.get("required")
    if is_blank(required):
        required = bool(not is_blank(voltage) or power > 0)
        electrical_required_source = "inferred_from_explicit_electrical_spec"
    else:
        required = bool(required)
        electrical_required_source = "source"

    p["electrical"] = {
        "required": required,
        "required_source": electrical_required_source,
        "voltage": clean_text(voltage),
        "power_w": power,
        "power_available": power > 0
    }

    # ----------------------------------------
    # Explicit quality flags
    # ----------------------------------------
    p["spatial_validation_possible"] = all(
        p["dimensions"][field] > 0
        for field in ["width_mm", "depth_mm", "height_mm"]
    )

    p["data_quality"] = (
        "complete_for_spatial_use"
        if p["spatial_validation_possible"]
        else "complete_with_unspecified_source_fields"
    )

    cleaned_products.append(p)

print(f"Cleaned {len(cleaned_products)} products.")


Cleaned 36 products.


In [29]:
# ============================================
# 5. VERIFY THERE ARE NO NULL/NaN VALUES
# ============================================

def find_missing_paths(obj, path=""):
    """Recursively find actual missing values in a product dictionary.

    Empty lists and empty dictionaries are VALID MongoDB values and are
    therefore not treated as missing. Only None/NaN/pd.NA and blank strings
    are considered missing.
    """
    missing = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            child_path = f"{path}.{key}" if path else key
            missing.extend(find_missing_paths(value, child_path))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            missing.extend(find_missing_paths(value, f"{path}[{i}]"))

    else:
        if is_blank(obj):
            missing.append(path)

    return missing


remaining_missing = []

for product in cleaned_products:
    problems = find_missing_paths(product)
    if problems:
        remaining_missing.append({
            "product_id": product.get("product_id"),
            "missing_paths": problems
        })

print("Products with remaining NULL/NaN/blank values:", len(remaining_missing))

if remaining_missing:
    display(pd.DataFrame(remaining_missing))
else:
    print("✓ No None, NaN, pd.NA, or blank strings remain.")


Products with remaining NULL/NaN/blank values: 0
✓ No None, NaN, pd.NA, or blank strings remain.


In [30]:
# ============================================
# 6. CREATE CLEAN INSPECTION DATAFRAME
# ============================================

clean_inspection_data = []

for p in cleaned_products:
    clean_inspection_data.append({
        "product_id": p["product_id"],
        "product_name": p["product_name"],
        "category": p["category"],
        "subcategory": p["subcategory"],
        "collection": p["collection"],
        "width_mm": p["dimensions"]["width_mm"],
        "depth_mm": p["dimensions"]["depth_mm"],
        "height_mm": p["dimensions"]["height_mm"],
        "width_available": p["dimensions"]["width_mm_available"],
        "depth_available": p["dimensions"]["depth_mm_available"],
        "height_available": p["dimensions"]["height_mm_available"],
        "rough_in_mm": p["installation"]["rough_in_mm"],
        "rough_in_available": p["installation"]["rough_in_available"],
        "installation_type": p["installation"]["type"],
        "waste_outlet": p["installation"]["waste_outlet"],
        "electrical_required": p["electrical"]["required"],
        "voltage": p["electrical"]["voltage"],
        "power_w": p["electrical"]["power_w"],
        "power_available": p["electrical"]["power_available"],
        "material": p["material"],
        "color": ", ".join(p["color"]),
        "feature_count": len(p["features"]),
        "spatial_validation_possible": p["spatial_validation_possible"],
        "data_quality": p["data_quality"]
    })

clean_inspection_df = pd.DataFrame(clean_inspection_data)

print("Clean dataset shape:", clean_inspection_df.shape)
display(clean_inspection_df.head())


Clean dataset shape: (36, 24)


,product_id,product_name,category,subcategory,collection,width_mm,depth_mm,height_mm,width_available,depth_available,...,waste_outlet,electrical_required,voltage,power_w,power_available,material,color,feature_count,spatial_validation_possible,data_quality
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),Other,Not specified,Not specified,344.0,483.0,141.0,True,True,...,Not specified,False,Not specified,-1.0,False,Vitreous China,White,6,True,complete_for_spatial_use
1,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Vessel,Vessel,Not specified,470.0,116.0,139.0,True,True,...,Not specified,False,Not specified,-1.0,False,Vitreous China,White,5,True,complete_for_spatial_use
2,1311520-A04-D,Mica® Square 393mm Basin,BASINS / COUNTERTOP,Countertop Basin,Not specified,393.0,393.0,-1.0,True,True,...,DN32 waste without overflow,False,Not specified,-1.0,False,Ceramic,"white, honed black",3,False,complete_with_unspecified_source_fields
3,K-12927IN,Complimentary™ Hygiene Spray,Other,Not specified,Not specified,-1.0,-1.0,-1.0,False,False,...,Not specified,False,Not specified,-1.0,False,Not specified,"Polished Chrome, Vibrant® French Gold, Vibrant...",5,False,complete_with_unspecified_source_fields
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,One-piece elongated toilet,Veil,-1.0,-1.0,390.0,False,False,...,Floor,False,Not specified,-1.0,False,Plastic seat,White,13,False,complete_with_unspecified_source_fields


In [31]:
# ============================================
# 7. FINAL NULL CHECK USING PANDAS
# ============================================

null_counts = clean_inspection_df.isna().sum()
blank_counts = (clean_inspection_df.astype(str).apply(lambda col: col.str.strip() == "")).sum()

final_quality_report = pd.DataFrame({
    "null_count": null_counts,
    "blank_string_count": blank_counts
})

display(final_quality_report)

assert null_counts.sum() == 0, "ERROR: Null values still exist."
assert blank_counts.sum() == 0, "ERROR: Blank strings still exist."

print("✓ FINAL CHECK PASSED — no missing values remain in the cleaned inspection dataset.")


,null_count,blank_string_count
product_id,0,0
product_name,0,0
category,0,0
subcategory,0,0
collection,0,0
width_mm,0,0
depth_mm,0,0
height_mm,0,0
width_available,0,0
depth_available,0,0


✓ FINAL CHECK PASSED — no missing values remain in the cleaned inspection dataset.


In [32]:
# ============================================
# 8. SAVE CLEANED DATASET TO CSV
# ============================================

clean_csv_path = Path("kohler_cleaned_products.csv")
clean_inspection_df.to_csv(clean_csv_path, index=False)

print("Cleaned CSV saved to:")
print(clean_csv_path.resolve())


Cleaned CSV saved to:
C:\Users\Abhist\Desktop\KOHLER\scripts\kohler_cleaned_products.csv


In [33]:
# ============================================
# 9. REPLACE PRODUCTS COLLECTION WITH CLEAN DATA
# ============================================

# The backup created earlier remains untouched.
# This cell updates only the main products collection.

products_collection.delete_many({})

if cleaned_products:
    products_collection.insert_many(cleaned_products)

print(f"✓ MongoDB updated with {len(cleaned_products)} cleaned products.")


✓ MongoDB updated with 36 cleaned products.


In [34]:
# ============================================
# 10. VERIFY MONGODB AFTER CLEANING
# ============================================

mongo_products = list(products_collection.find({}, {"_id": 0}))

mongo_missing = []

for product in mongo_products:
    problems = find_missing_paths(product)

    if problems:
        mongo_missing.append({
            "product_id": product.get("product_id", "Unknown"),
            "missing_paths": problems
        })

print("MongoDB product count:", len(mongo_products))
print("MongoDB products with NULL/NaN/blank values:", len(mongo_missing))

if mongo_missing:
    print("\nProblematic MongoDB documents:")
    display(pd.DataFrame(mongo_missing))

    # Do not silently continue when a real missing value exists.
    raise AssertionError(
        "ERROR: NULL/NaN/blank values remain in MongoDB. "
        "See the table above for the exact fields."
    )

print("✓ MongoDB CLEANING VERIFIED — no NULL/NaN/blank values remain.")


MongoDB product count: 36
MongoDB products with NULL/NaN/blank values: 0
✓ MongoDB CLEANING VERIFIED — no NULL/NaN/blank values remain.


## Important interpretation of `-1`

For numeric specifications, `-1` means **the source data did not provide that specification**. It is deliberately not replaced with `0` or a guessed dimension.

Use the availability flags before spatial calculations:

```python
if product["dimensions"]["depth_mm_available"]:
    # Safe to use depth_mm for a spatial calculation
else:
    # Do not use the unknown depth as a real measurement
```

The cleaning process guarantees that actual database NULL/NaN/blank values are removed.

**Note:** Empty lists (`[]`) and empty dictionaries (`{}`) are valid MongoDB values and are **not** considered missing. For example, an empty `validation_problems` list simply means no validation problems were recorded.
